# 第 9 讲实验：Scaling Laws 与 Compute Budget

本 Notebook 使用代码生成的 synthetic observations，验证 power-law 拟合与 residual 诊断流程。数据不是实际模型训练观测，也不是 Kaplan/Chinchilla 论文数据。

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

START = Path.cwd().resolve()
REPO_ROOT = next((p for p in (START, *START.parents) if (p / "code" / "lecture_09").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("请在课程仓库内启动 Jupyter Lab")
sys.path.insert(0, str(REPO_ROOT / "code" / "lecture_09"))
from scaling_laws_lab import (
    compute_budget_tradeoff, fit_power_law, synthetic_power_law_data,
    tokens_for_compute_budget, training_flops_estimate,
)

print("PyTorch:", torch.__version__)
print("All observations below are synthetic, generated for method validation.")

## 实验 1：固定 loss floor 的幂律拟合

拟合形式为 L(x)=L_floor + A·x^(-alpha)。为了避免 floor、amplitude 和 exponent 同时拟合时的弱可辨识性，本实验把 floor 作为显式输入。

In [ ]:
x = torch.logspace(2, 8, steps=18, dtype=torch.float64).tolist()
data = synthetic_power_law_data(
    x, floor=1.2, amplitude=25.0, exponent=0.22,
    relative_noise=0.01, seed=9,
)
fit = fit_power_law(data["x"], data["loss"], floor=1.2)
print(f"sample count: {fit['sample_count']}")
print(f"known synthetic exponent: 0.2200")
print(f"fitted exponent: {fit['exponent']:.4f}")
print(f"fitted amplitude: {fit['amplitude']:.4f}")
print(f"RMSE (original loss scale): {fit['rmse']:.6f}")
print(f"R² (original loss scale): {fit['r_squared']:.6f}")
assert fit["rmse"] >= 0 and torch.isfinite(torch.tensor(fit["exponent"])).item()

## 实验 2：在 log-log 空间观察 excess loss

若 floor 已知，应绘制 L-L_floor 与规模 x 的关系；直接对原始 L 取对数时，非零 floor 会使曲线在大规模处偏离直线。

In [ ]:
pred = fit["predictions"]
excess_loss = [y - 1.2 for y in data["loss"]]
pred_excess = [y - 1.2 for y in pred]
order = sorted(range(len(x)), key=lambda i: x[i])
plt.figure(figsize=(7, 4.5))
plt.loglog([x[i] for i in order], [excess_loss[i] for i in order], "o", label="synthetic observations")
plt.loglog([x[i] for i in order], [pred_excess[i] for i in order], "-", label="fitted power law")
plt.xlabel("Scale x (synthetic)")
plt.ylabel("Loss - fixed floor")
plt.title("Power-law fit in log-log space")
plt.legend()
plt.grid(True, which="both", alpha=0.25)
plt.tight_layout()
plt.show()

## 实验 3：残差诊断

高 R² 不保证外推可靠。残差是否随规模呈现系统趋势，常比单一拟合分数更能提示模型失配、数据分布变化或未控制的训练 recipe。

In [ ]:
residuals = fit["residuals"]
plt.figure(figsize=(7, 4))
plt.axhline(0.0, linestyle="--")
plt.semilogx(data["x"], residuals, "o-")
plt.xlabel("Scale x (synthetic, log axis)")
plt.ylabel("Observed loss - fitted loss")
plt.title("Residuals on original loss scale")
plt.grid(True, which="both", alpha=0.25)
plt.tight_layout()
plt.show()
print("Residual range:", min(residuals), "to", max(residuals))

## 实验 4：固定计算预算的 N/D 约束

C≈6ND 给出一个粗略 iso-FLOP 约束。它只能说明更大的 N 会挤占 D；若没有观测 loss 或经过验证的联合损失模型，不能单靠这条曲线判断最优配置。

In [ ]:
compute_budget = 1e20
candidates = [0.5e9, 1e9, 2e9, 4e9]
allocations = compute_budget_tradeoff(compute_budget, candidates)
for item in allocations:
    print(
        f"N={item['parameter_count']:.2e}, "
        f"D≈{item['token_count']:.2e} tokens, "
        f"C≈{item['estimated_compute_flops']:.2e} FLOPs"
    )
plt.figure(figsize=(6.5, 4))
plt.loglog(
    [item["parameter_count"] for item in allocations],
    [item["token_count"] for item in allocations],
    "o-",
)
plt.xlabel("Parameter count N")
plt.ylabel("Implied token count D")
plt.title("Iso-FLOP constraint: C ≈ 6ND (not an optimum)")
plt.grid(True, which="both", alpha=0.25)
plt.tight_layout()
plt.show()
assert all(abs(item["estimated_compute_flops"] / compute_budget - 1) < 1e-12 for item in allocations)

## 复盘

1. 为什么固定 floor 会让幂律拟合更稳定，但也引入一个显式假设？
2. 为什么 log-space least squares 与 original-space RMSE 不是同一个优化目标？
3. 怎样的残差形状提示单一幂律不够用？
4. 为什么 iso-FLOP 曲线不是 compute-optimal 曲线？
5. 论文中的最优配比为什么不能直接当作所有模型的固定常数？